# Virchow Training on Google Colab (Resume-Safe)

Runs `scripts/train_virchow_preprocessed_colab.py` with **checkpoints on Google Drive** (resume after disconnect).

After you set `PREPROCESSED_DIR` on Drive, the next cell **copies** those `.h5` files to **`/content/local_preprocessed_h5`** (fast local SSD) and repoints `PREPROCESSED_DIR` so training reads local data. **Outputs/checkpoints** still go to `RUN_DIR` on Drive.

**Also logs:** ROC-AUC, PR-AUC, Brier, log loss, ECE (15 bins); **temperature scaling** on val; optional **MC Dropout** uncertainty at the end. **Outputs** under `RUN_DIR/`: `metrics_history.json`, `metrics_epoch_XXX.json`, `metrics_final_detailed.json`, `temperature_fit.json`, `val_predictions.npz`, checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/<your-username>/<your-repo>.git GP_ECG
%cd /content/GP_ECG

If you already uploaded your repo into Drive instead of cloning, skip previous cell and set `REPO_DIR` below to that path.

In [ ]:
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_preprocessed_run_01'
HF_TOKEN = ''  # optional: paste token if Virchow access is gated

import os
os.makedirs(RUN_DIR, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

In [ ]:
import os
import shutil

# Copy preprocessed .h5 from Google Drive → Colab local SSD (faster reads). RUN_DIR stays on Drive.
LOCAL_PREPROCESSED_DIR = "/content/local_preprocessed_h5"

_src = PREPROCESSED_DIR
if _src.startswith("/content/drive/"):
    if not os.path.isdir(_src):
        raise FileNotFoundError("Drive path not found — mount Drive and check PREPROCESSED_DIR:\n  " + _src)
    if not os.path.exists(LOCAL_PREPROCESSED_DIR):
        print("Copying preprocessed data Drive → local SSD (may take several minutes)...")
        print("  from:", _src)
        shutil.copytree(_src, LOCAL_PREPROCESSED_DIR)
        print("  done:", LOCAL_PREPROCESSED_DIR)
    else:
        print("Using existing local copy:", LOCAL_PREPROCESSED_DIR)
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
else:
    print("PREPROCESSED_DIR is not under /content/drive/ — using as-is:", _src)

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn

Optional if model access is gated: run login once and paste your Hugging Face token.

In [ ]:
# !huggingface-cli login

In [ ]:
%cd {REPO_DIR}
# --head-dropout 0.2 enables MC Dropout; set --mc-samples 0 to skip slow MC pass at end
# --no-save-val-preds saves disk (skip val_predictions.npz); metrics JSON still saved
!python scripts/train_virchow_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 64 \
  --num-workers 2 \
  --head-dropout 0.2 \
  --mc-samples 30 \
  --resume \
  --save-every-epoch-copy

In [ ]:
import json, os
print('Run dir:', RUN_DIR)
artifacts = [
    'checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json',
    'metrics_final_detailed.json', 'temperature_fit.json', 'run_config.json', 'run_manifest.json',
    'run_progress.json', 'val_predictions.npz',
]
for name in artifacts:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

hist = os.path.join(RUN_DIR, 'metrics_history.json')
if os.path.exists(hist):
    with open(hist, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    if rows:
        print('Last epoch record:', rows[-1])

det = os.path.join(RUN_DIR, 'metrics_final_detailed.json')
if os.path.exists(det):
    with open(det, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('metrics_final_detailed keys:', list(d.keys()))